# Prediction-market screener — explorationOpen-ended analysis over the **full** dataset: every market, every snapshot,tradeable on Predict or not. The ranked shortlist is only one view of this data.**Framing.** In a prediction market, "high profit potential" and "high risk" are thesame variable. A contract at \$0.20 pays 5x precisely because the market thinks itprobably won't happen. There is no screen for "high profit, low risk". The only realedge is *disagreeing with the market price for a defensible reason* — which is what`edge` measures, and only where a fair-value model exists.Everything below is **analysis, not advice**. Execution is manual, in theWealthsimple Predict app.

In [ ]:
# Run this notebook from the repository root.import pandas as pdimport matplotlib.pyplot as pltfrom screener.analysis.queries import Analysispd.set_option("display.max_columns", 60)pd.set_option("display.width", 200)a = Analysis("data/screener.db")a.summary()

## 1. What's in the datasetRun history and table sizes — check this first if numbers look stale.

In [ ]:
a.runs(limit=10)[["run_id", "started_at", "status", "markets_seen",                  "markets_tradeable", "api_requests", "error_count"]]

## 2. All markets, not just the shortlist`markets()` with no arguments returns everything ingested. Every filter is optional.

In [ ]:
all_markets = a.markets()print(f"{len(all_markets)} markets, {all_markets['tradeable'].sum():.0f} tradeable on Predict")all_markets.groupby("category", dropna=False).agg(    n=("ticker", "count"),    tradeable=("tradeable", "sum"),    median_spread=("spread", "median"),    median_volume=("volume", "median"),).sort_values("n", ascending=False)

### Why contracts were excludedUse this to tune `predict.series_prefix_allowlist` in `config.yaml`.

In [ ]:
excluded = all_markets[all_markets["tradeable"] == 0]excluded["not_tradeable_reason"].str.split("(").str[0].str.strip().value_counts().head(10)

In [ ]:
# Series that were excluded purely for not being on the allowlist. If you can# trade any of these in the Predict app, add its prefix to config.yaml.missing = excluded[excluded["not_tradeable_reason"].str.contains("allowlist", na=False)]missing.groupby("series_ticker").agg(    n=("ticker", "count"), category=("category", "first"), example=("title", "first")).sort_values("n", ascending=False).head(25)

## 3. Edge distributionEdge is `model_prob - implied_price`, defined only where a fair-value model coversthe contract. A tight cluster around zero means the models agree with the market —which is the normal and expected state. Fat tails deserve scrutiny of the *model*before the market.

In [ ]:
edges = a.edge_distribution()if edges.empty:    print("No modelled contracts yet. Enable models in config.yaml and re-run signals.")else:    display(edges.head(15))    fig, ax = plt.subplots(figsize=(9, 3.5))    ax.hist(edges["edge"].dropna(), bins=30, color="#3b5bdb", alpha=.8)    ax.axvline(0, color="#a8322a", lw=1.2, ls="--")    ax.set_xlabel("edge  (model − market)"); ax.set_ylabel("contracts")    ax.set_title("Edge distribution — where do we disagree with the market?")    plt.tight_layout(); plt.show()

## 4. Spread vs liquidityWide spreads and thin books are a **risk** (hard to enter, harder to exit) *and* apotential opportunity (directional liquidity provision). Bubble size is open interest.

In [ ]:
sl = a.spread_vs_liquidity(tradeable_only=False)if sl.empty:    print("No snapshot data yet.")else:    fig, ax = plt.subplots(figsize=(9, 5))    sc = ax.scatter(sl["volume"].clip(lower=1), sl["spread"].clip(lower=0),                    s=(sl["open_interest"].clip(lower=1) ** 0.5),                    c=sl["implied_prob"], cmap="coolwarm", alpha=.65, edgecolors="none")    ax.set_xscale("log")    ax.set_xlabel("volume (log)"); ax.set_ylabel("spread, cents")    ax.set_title("Spread vs liquidity — colour = implied probability")    plt.colorbar(sc, ax=ax, label="implied prob")    plt.tight_layout(); plt.show()

## 5. Price history for any tickerCombines stored snapshots with Kalshi candlesticks.

In [ ]:
# Pick the highest-scoring contract from the latest run, or set one by hand.top = a.signals(limit=1)ticker = top["ticker"].iloc[0] if not top.empty else Noneprint("plotting:", ticker)if ticker:    hist = a.price_history(ticker)    if not hist.empty:        fig, ax = plt.subplots(figsize=(10, 4))        for source, group in hist.groupby("source"):            ax.plot(group["ts"], group["implied_prob"], marker=".", lw=1.2, label=source)        ax.set_ylim(0, 1); ax.legend()        ax.set_ylabel("implied probability"); ax.set_title(ticker)        plt.tight_layout(); plt.show()    display(a.trades(ticker, limit=10))

## 6. Favorite–longshot calibrationThe classic retail bias: longshots are systematically overpriced and favoritesunderpriced. This buckets settled contracts by their price ~24h before close andcompares each bucket against how often it actually resolved YES.`bias < 0` in a bucket means contracts at that price resolved YES **less** often thanthe price implied — i.e. buyers overpaid.Needs settled contracts, so it stays empty until the database has run for a while.

In [ ]:
calib = a.calibration(bins=10, hours_before_close=24)if calib.empty:    print("No settled contracts with pre-close snapshots yet — keep the cron running.")else:    display(calib)    fig, ax = plt.subplots(figsize=(5.5, 5.5))    ax.plot([0, 1], [0, 1], ls="--", c="#888", lw=1, label="perfectly calibrated")    ax.scatter(calib["mean_implied"], calib["realized_yes_rate"],               s=calib["n"] * 6, c="#3b5bdb", alpha=.75)    ax.set_xlabel("mean implied probability"); ax.set_ylabel("realised YES rate")    ax.set_title("Calibration — bubble size = sample count"); ax.legend()    plt.tight_layout(); plt.show()

## 7. Signal backtest on settled contractsCoarse check that a signal separates outcomes at all.**This is not a strategy backtest.** It ignores execution, slippage, and the fact thatyou trade manually, hours after the snapshot, at a different price. Treat a positiveresult as "worth investigating", never as a validated edge.

In [ ]:
for signal in ("edge_flag", "longshot_flag", "spread_flag", "liquidity_flag"):    result = a.signal_backtest(signal)    print(f"\n--- {signal} ---")    print("no settled contracts yet" if result.empty else result.to_string(index=False))

## 8. Free-form queries`a.sql()` runs any SELECT. The database is opened read-only, so explorationcannot mutate it.

In [ ]:
a.sql("""    SELECT m.series_ticker,           COUNT(*)                        AS contracts,           ROUND(AVG(s.spread), 2)         AS avg_spread_cents,           ROUND(AVG(s.mid_price), 1)      AS avg_price_cents,           SUM(s.volume)                   AS total_volume      FROM markets m      JOIN (SELECT s1.* FROM snapshots s1            JOIN (SELECT ticker, MAX(ts) mts FROM snapshots GROUP BY ticker) s2              ON s1.ticker = s2.ticker AND s1.ts = s2.mts) s        ON s.ticker = m.ticker     GROUP BY m.series_ticker     HAVING contracts > 1     ORDER BY total_volume DESC     LIMIT 20""")

### Custom screensCompose the query helpers however you like — cheap contracts closing soon with atight book, say. Remember what a cheap contract *is*: the market's judgement that itprobably won't happen.

In [ ]:
cheap_and_soon = a.signals(    tradeable_only=True,    price_band=(0.05, 0.30),    max_spread_cents=3,    max_days_to_close=14,)cols = ["ticker", "title", "implied_prob", "model_prob", "edge",        "ev_per_contract", "annualized_if_win", "days_to_close", "volume", "score"]cheap_and_soon[[c for c in cols if c in cheap_and_soon.columns]].head(20)

## 9. Export for external tools

In [ ]:
from screener.analysis.export import export_dataset# export_dataset("data/screener.db", "exports", "parquet")print("uncomment to export CSV/Parquet to ./exports")

---**Reminder.** Every number above describes a contract; none of it recommends one.Before acting on anything here, read the contract's full settlement rules and closetime, confirm it is actually listed in the Predict app, and remember that contractssettle in **USD**.